In [25]:
import rasterio
from dist_s1.dist_plot import get_dist_s1_mpl_cmap
import matplotlib.pyplot as plt
import numpy as np
from dem_stitcher.rio_window import read_raster_from_window
from dist_s1_enumerator.mgrs_burst_data import get_mgrs_table
import pandas as pd
from pathlib import Path
from tile_mate import get_raster_from_tiles
from dem_stitcher.rio_tools import reproject_arr_to_match_profile

import earthaccess
from dist_s1_enumerator.mgrs_burst_data import get_mgrs_table
import pandas as pd
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential
from pathlib import Path

from functools import partial
import concurrent.futures


In [26]:
df_mgrs = get_mgrs_table()

In [27]:
earthaccess.login()

In [28]:
START_TIME = pd.Timestamp('2024-01-01')
STOP_TIME = pd.Timestamp('2025-01-01')


In [29]:
val_dist_s1_status_dir = Path('val_products_transformer_optimized-max10_processed_2026-02-05')
ts_dirs = list(val_dist_s1_status_dir.glob('*/'))

In [30]:
mgrs_tile_ids = [x.stem.split('__')[1] for x in ts_dirs]
mgrs2tsid = {d.stem.split('__')[1]: d.stem for d in ts_dirs}
mgrs_tile_ids[:3], list(mgrs2tsid.items())[:3]

(['20KNG', '17MPT', '30SUD'],
 [('20KNG', 'treelosswet__20KNG'),
  ('17MPT', 'waternew__17MPT'),
  ('30SUD', 'other__30SUD')])

In [31]:
all_dist_hls_dir = Path('dist_hls/validation_sites')
all_dist_hls_dir.mkdir(exist_ok=True, parents=True)

# Testing

In [ ]:
def get_opera_id(query_item) -> str:
    return dict(query_item.__dict__['render_dict'])['meta']['native-id']

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, max=10))
def query_dist_hls_cmr_for_status_links(mgrs_tile_id: str, start_time=str(START_TIME.date()), stop_time=str(STOP_TIME.date())):
    collection_short_name = 'OPERA_L3_DIST-ALERT-HLS_V1'
    mgrs_bounds = tuple(df_mgrs[df_mgrs.mgrs_tile_id == mgrs_tile_id].total_bounds)
    mgrs_bounds = tuple(float(x) for x in mgrs_bounds)
    datasets_found = earthaccess.search_data(
        short_name=collection_short_name,
        temporal=(start_time, stop_time),
        cloud_hosted=True,
        bounding_box=mgrs_bounds
    )
    datasets_found = [d for d in datasets_found if mgrs_tile_id in get_opera_id(d)]
    links_status = [link for r in datasets_found for link in r.data_links() if 'DIST-STATUS.tif' in link.split('/')[-1]]
    links_veg_anom = [link for r in datasets_found for link in r.data_links() if 'VEG-ANOM.tif' in link.split('/')[-1]]
    links = links_status + links_veg_anom
    return links
    
def get_status_links_formatted_output(mgrs_tile_id: str) -> dict:
    links = query_dist_hls_cmr_for_status_links(mgrs_tile_id)
    out = {'mgrs_tile_id': mgrs_tile_id, 'links': links}
    return out

def get_opera_id_from_link(link: str) -> str:
    opera_id_and_layer = link.split('/')[-1]
    opera_id = '_'.join(opera_id_and_layer.split('_')[:-1])
    return opera_id

def get_processing_time_from_link(link: str) -> pd.Timestamp:
    opera_id_and_layer = link.split('/')[-1]
    processing_time = pd.Timestamp(opera_id_and_layer.split('_')[5])
    return processing_time

In [33]:
links = query_dist_hls_cmr_for_status_links(mgrs_tile_ids[1])
links[:3]


['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/OPERA_L3_DIST-ALERT-HLS_V1/OPERA_L3_DIST-ALERT-HLS_T17MPT_20240103T152657Z_20240207T214546Z_L9_30_v1/OPERA_L3_DIST-ALERT-HLS_T17MPT_20240103T152657Z_20240207T214546Z_L9_30_v1_VEG-ANOM.tif',
 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/OPERA_L3_DIST-ALERT-HLS_V1/OPERA_L3_DIST-ALERT-HLS_T17MPT_20240104T155221Z_20240207T214621Z_S2A_30_v1/OPERA_L3_DIST-ALERT-HLS_T17MPT_20240104T155221Z_20240207T214621Z_S2A_30_v1_VEG-ANOM.tif',
 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/OPERA_L3_DIST-ALERT-HLS_V1/OPERA_L3_DIST-ALERT-HLS_T17MPT_20240106T153629Z_20240207T214656Z_S2B_30_v1/OPERA_L3_DIST-ALERT-HLS_T17MPT_20240106T153629Z_20240207T214656Z_S2B_30_v1_VEG-ANOM.tif']

In [ ]:
d = get_status_links_formatted_output(mgrs_tile_ids[1])

# Automate

In [35]:
# data_dict = list(map(get_status_links_formatted_output, tqdm(mgrs_tile_ids[:3])))

In [36]:
l = mgrs_tile_ids[:]
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    data_dicts = list(tqdm(executor.map(get_status_links_formatted_output, l), total=len(l)))


  0%|          | 0/85 [00:00<?, ?it/s]

100%|██████████| 85/85 [01:05<00:00,  1.30it/s]


In [37]:
@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=4, max=10))
def download_one(hls_link: str, dst_dir: Path) -> str:
    opera_id = get_opera_id_from_link(hls_link)
    out_dir = dst_dir / opera_id
    out_dir.mkdir(exist_ok=True, parents=True)
    r = earthaccess.download(hls_link, out_dir, pqdm_kwargs={'disable': True})
    return r

def download_data_dist_hls_data(data_dict: dict) -> str:
    links = data_dict['links']
    mgrs_tile_id = data_dict['mgrs_tile_id']
    ts_id = mgrs2tsid[mgrs_tile_id]
    ts_dir = all_dist_hls_dir / ts_id
    if links:
        links = sorted(links, key=get_processing_time_from_link)
        download_one_partial = partial(download_one, dst_dir=ts_dir)
        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
            out = list(tqdm(executor.map(download_one_partial, links), total=len(links), disable=True))
        return out
    else:
        return []

In [ ]:
out_lists = [download_data_dist_hls_data(data_dict) for data_dict in tqdm(data_dicts[:])]

 12%|█▏        | 10/85 [10:06<1:14:09, 59.33s/it]